---
authors: Freek Pols
updated: December 3, 2025
---

# $pV-$ diagram

## Introductie

Wanneer je een gas snel samenperst bij gelijkblijvende volume neemt de temperatuur toe. Wanneer je het gas vervolgens laat ontsnappen, neemt de temperatuur af omdat het gas arbeid verricht. 

We gaan eigenschappen van dit proces bestuderen. In dit practicum ga je een $p-t$-diagram van een brandblusser bestuderen en gebruiken om de specifieke warmte verhouding $\gamma$ te bepalen voor lucht.

## Experiment (60 min)

In dit experiment vullen we een brandblusser met lucht ($P_1, T_1=T_{atm}$). We laten de lucht snel ontsnappen ($P_2=P_{atm}, T_2$), in zo'n korte tijd dat we aannemen dat dit een adiabatisch proces is. Doordat het gas arbeid verricht zal het gas afkoelen. Wanneer we dan, kort na het ontsnappen van de lucht, de kraan weer dicht doen, zal de druk weer toenemen ($P_3,T_3=T_{atm}$). 

In het eerste deel van het proces geldt: 

$$
  T_1^\gamma P_1^{1-\gamma} = T_2^\gamma P_2^{1-\gamma}
$$

ook wel bekend als ... , met $\gamma$ de specifieke warmte verhouding: $\gamma=\frac{C_p}{C_V}$.

Het tweede deel van het proces kan beschreven worden met de wet van Gay-Lussac:

$$ 
  \frac{P_2}{T_2} = \frac{P_3}{T_3}
$$


Onder de aanname dat $T_1 = T_3 = T_{atm}$ volgt hieruit:

$$
 \gamma=\frac{\ln{P_1}-\ln{P_{atm}}}{\ln{P_1}-\ln{P_3}}
$$

```{exercise}
1. Zet de druksensor in het brandblusapparaat, zet de kraan er op en vul de fles met lucht. 
2. Wacht een tijd (~30 min). Waarom?
3. Knijp in de hendel zodat de lucht ontsnapt. Zodra het lucht niet meer ontsnapt, laat de hendel los zodat er geen uitwisseling van lucht meer is.
4. Wacht een korte tijd en haal dan je Arduino en de sd-kaart uit de blusser.
5. Lees de waarden uit in je eigen python script en bepaalde waarde van $\gamma$. Vergelijk deze met de literatuurwaarde ($\gamma = 1.45$)


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

dataset = np.loadtxt('drukmetingenbrand.csv', delimiter=',', skiprows=1)

time= dataset[:, 0]/1000  # convert ms to s
P_bit = dataset[:, 1] # bit format

def pressure_convertor(bits):
    # linear interpolatie tussen kalibratiepunten
    bits_calib = np.array([180, 222, 314, 150, 530])
    P_calib = np.array([101.325, 101.325/0.75, 202.65, 101.325/1.25, 101.325/0.25])  # in kPa
    P = np.interp(bits, bits_calib, P_calib)
    return P*1000  # in Pa

def gamma_calculator(P1, Patm, P3):
    return (np.log(P1/Patm) / np.log(P1/P3))


def plot_with_slider(t_sel):
    idx = np.argmin(np.abs(time - t_sel))
    t = time[idx]
    p = P_bit[idx]
    plt.figure(dpi=300)
    plt.plot(time, P_bit, label='P vs t diagram')
    plt.axvline(t, color='gray', ls='--', lw=0.8)
    plt.plot(t, p, 'ro')
    plt.xlabel('tijd (s)')
    plt.ylabel('Pressure (bits)')
    plt.title('P-t diagram (kies tijd met slider)')
    plt.grid(True)
    plt.legend()
    plt.show()

# stapgrootte op basis van dataresolutie
step = float(np.median(np.diff(np.unique(time)))) if len(time) > 1 else 0.01
interact(plot_with_slider,
         t_sel=FloatSlider(value=time.mean(), min=float(time.min()), max=float(time.max()),
                           step=step, description='tijd (s)'))

P1 = pressure_convertor(992)
print(P1)
Patm = 101325
P3 = pressure_convertor(216)
print(P1)
gamma = gamma_calculator(P1, Patm, P3)
print(f'Gamma = {gamma:.2f}')

print(1-gamma/1.45)

interactive(children=(FloatSlider(value=1178.6589999999999, description='tijd (s)', max=2355.409, min=1.909, s…

405300.0
405300.0
Gamma = 1.22
0.15763702161416027


Onze berekende $\gamma = 1.22$  wat een afwijking is van ongeveer 16%, onder de aannames die zijn gemaakt is dit een behoorlijk resultaat.